# MHBAP — TCMT Training (Google Colab GPU)

**Model:** Temporal Cross-Modal Transformer (TCMT)  
**Input:** `(B, T, 58)` — 5 modalities (face/gaze/pose/voice/hci)  
**Outputs:** emotion 4-class, stress, engagement, attention, fatigue  
**Arch:** D_MODEL=128, N_HEADS=4, N_LAYERS=3, FFN=256  
**Datasets:** FER2013 + RAF-DB + WESAD (HuggingFace streaming)

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom

## 1 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Runtime → Change runtime type → T4 GPU')
device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/MHBAP_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Checkpoint dir: {DRIVE_DIR}')

## 3 — Install dependencies

In [ ]:
%%capture
!pip install -q datasets Pillow scikit-learn scipy
print('done')

## 4 — Clone repo (private — requires GitHub PAT)

You will be prompted for a **GitHub Personal Access Token** with `Contents: Read-only` scope.
The token is used only to authenticate the `git clone` command and is **never stored, printed, or written to disk**.

To generate one: GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens → New token  
Permissions needed: **Repository: Contents → Read-only**

In [ ]:
import os, subprocess, getpass

REPO_OWNER  = 'ashhal-kaleem'
REPO_NAME   = 'MHBAP'
REPO_BRANCH = 'feature/production-hardening'
REPO_DIR    = '/content/MHBAP'

# Prompt for PAT — input is masked, never echoed or stored
token = getpass.getpass('GitHub PAT (Contents: Read-only scope): ')
if not token.strip():
    raise ValueError('No token entered. Cell cannot continue.')

# Build authenticated URL in memory only — never written to any file
AUTH_URL = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

try:
    if os.path.exists(REPO_DIR):
        # Re-set remote with token for pull, then immediately unset
        subprocess.run(['git','-C',REPO_DIR,'remote','set-url','origin', AUTH_URL],
                       check=True, capture_output=True)
        r = subprocess.run(['git','-C',REPO_DIR,'pull','origin',REPO_BRANCH],
                           capture_output=True, text=True)
        print(r.stdout.strip() or r.stderr.strip())
    else:
        r = subprocess.run(
            ['git','clone','--branch',REPO_BRANCH,'--depth','1', AUTH_URL, REPO_DIR],
            capture_output=True, text=True
        )
        # Print stderr only if clone failed (stdout is empty on success)
        if r.returncode != 0:
            raise RuntimeError(f'Clone failed: {r.stderr.strip()}')
        print('Clone successful.')
finally:
    # Always scrub the token from memory and reset remote to unauthenticated URL
    SAFE_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
    subprocess.run(['git','-C',REPO_DIR,'remote','set-url','origin', SAFE_URL],
                   capture_output=True)
    del token, AUTH_URL

os.chdir(REPO_DIR)
log = subprocess.run(['git','log','--oneline','-3'], capture_output=True, text=True)
print(log.stdout)
print('Remote after scrub:', subprocess.run(
    ['git','remote','get-url','origin'], capture_output=True, text=True).stdout.strip())

## 5 — Imports & smoke test

In [ ]:
import sys
if '/content/MHBAP' not in sys.path:
    sys.path.insert(0, '/content/MHBAP')
from ml.fusion.tcmt import TCMT, D_MODEL, N_HEADS, N_LAYERS, FFN_DIM, EMOTION_CLASSES
from ml.fusion.feature_vector import FEATURE_DIM
from ml.evaluation.metrics import compute_all_metrics
print(f'FEATURE_DIM={FEATURE_DIM}  EMOTION_CLASSES={EMOTION_CLASSES}')
print(f'D_MODEL={D_MODEL}  N_HEADS={N_HEADS}  N_LAYERS={N_LAYERS}  FFN={FFN_DIM}')

## 6 — Resume checkpoint from Drive (if exists)

In [ ]:
import shutil, os
DRIVE_DIR   = '/content/drive/MyDrive/MHBAP_checkpoints'
WEIGHT_DIR  = '/content/MHBAP/ml/models/weights'
DRIVE_CKPT  = os.path.join(DRIVE_DIR,  'tcmt_trained.pt')
LOCAL_CKPT  = os.path.join(WEIGHT_DIR, 'tcmt_trained.pt')
os.makedirs(WEIGHT_DIR, exist_ok=True)
if os.path.exists(DRIVE_CKPT):
    shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
    sz = os.path.getsize(LOCAL_CKPT)/1024**2
    print(f'Resumed checkpoint from Drive ({sz:.2f} MB)')
elif os.path.exists(LOCAL_CKPT):
    print(f'Checkpoint in repo: {LOCAL_CKPT}')
else:
    print('No checkpoint found — training from scratch.')

## 7 — Config (edit hyperparameters here)

In [ ]:
CFG = dict(
    epochs        = 75,
    batch_size    = 128,
    lr            = 3e-4,
    weight_decay  = 1e-4,
    seed          = 42,
    fer_samples   = 9000,
    raf_samples   = 3000,
    wesad_samples = 2000,
    emo_loss_w    = 4.0,
    reg_loss_w    = 0.25,
    focal_gamma   = 2.0,
    label_smooth  = 0.05,
    weight_path   = '/content/MHBAP/ml/models/weights/tcmt_trained.pt',
    metrics_path  = '/content/MHBAP/ml/models/weights/tcmt_eval_metrics.json',
    drive_dir     = '/content/drive/MyDrive/MHBAP_checkpoints',
)
for k, v in CFG.items():
    print(f'  {k}: {v}')

## 8 — Load datasets

In [ ]:
import time, numpy as np
from ml.training.real_dataset import make_real_dataset
t0 = time.time()
print('Streaming datasets from HuggingFace...')
train_split, val_split, test_split = make_real_dataset(
    fer_samples=CFG['fer_samples'], raf_samples=CFG['raf_samples'],
    wesad_samples=CFG['wesad_samples'], seed=CFG['seed'],
)
print(f'Done in {time.time()-t0:.1f}s')
print(f'train={len(train_split["X"])}  val={len(val_split["X"])}  test={len(test_split["X"])}')
counts = np.bincount(train_split['emotion'], minlength=4)
print(f'Emotion class counts (train): {counts.tolist()}')

## 9 — Training loop (GPU)

In [ ]:
import json, os, shutil, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from ml.fusion.tcmt import TCMT, EMOTION_CLASSES
from ml.evaluation.metrics import compute_all_metrics

device = torch.device('cuda')

def _to_tensors(sp):
    return TensorDataset(
        torch.tensor(sp['X'],          dtype=torch.float32),
        torch.tensor(sp['emotion'],     dtype=torch.long),
        torch.tensor(sp['stress'],      dtype=torch.float32).unsqueeze(-1),
        torch.tensor(sp['engagement'],  dtype=torch.float32).unsqueeze(-1),
        torch.tensor(sp['attention'],   dtype=torch.float32).unsqueeze(-1),
        torch.tensor(sp['fatigue'],     dtype=torch.float32).unsqueeze(-1),
    )

def focal_ce(logits, tgt, gamma, weight, smooth, nc):
    s = torch.zeros_like(logits).fill_(smooth/(nc-1))
    s.scatter_(1, tgt.unsqueeze(1), 1.0-smooth)
    lp = F.log_softmax(logits, dim=-1)
    fw = (1 - torch.exp(lp))**gamma
    w  = weight.to(logits.device)[tgt].unsqueeze(1)
    return -(fw * s * lp * w).sum(dim=-1).mean()

def _fwd(m, X):
    if X.dim()==2: X=X.unsqueeze(1)
    B,T,F2 = X.shape
    tok = torch.cat([m.mod_proj(X[:,t,:]) for t in range(T)], dim=1)
    enc = m.encoder(torch.cat([m.cls_token.expand(B,-1,-1), tok], dim=1))
    h   = enc[:,0,:]
    return dict(emo=m.head_emotion(h),
                st=torch.sigmoid(m.head_stress(h)),
                en=torch.sigmoid(m.head_engagement(h)),
                at=torch.sigmoid(m.head_attention(h)),
                fa=torch.sigmoid(m.head_fatigue(h)))

def _eval(m, sp):
    m.eval()
    X = torch.tensor(sp['X'], dtype=torch.float32).to(device)
    with torch.no_grad():
        o = _fwd(m, X)
    tgt = {k: sp[k] for k in ('emotion','stress','engagement','attention','fatigue')}
    prd = {
        'emotion':    o['emo'].cpu().numpy(),
        'stress':     o['st'].cpu().numpy().squeeze(),
        'engagement': o['en'].cpu().numpy().squeeze(),
        'attention':  o['at'].cpu().numpy().squeeze(),
        'fatigue':    o['fa'].cpu().numpy().squeeze(),
    }
    return compute_all_metrics(tgt, prd)

# sampler
train_ds = _to_tensors(train_split)
emo_lbl  = train_split['emotion']
cc = np.bincount(emo_lbl, minlength=EMOTION_CLASSES).astype(float)
cc = np.where(cc==0, 1.0, cc)
sw = (1.0/cc)[emo_lbl]
sampler = torch.utils.data.WeightedRandomSampler(
    torch.tensor(sw, dtype=torch.float64), len(train_ds), replacement=True)
loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler,
                    num_workers=2, pin_memory=True)

ew = torch.tensor(1.0/cc, dtype=torch.float32)
ew = ew / ew.sum() * EMOTION_CLASSES

model = TCMT().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

if os.path.exists(CFG['weight_path']):
    ck = torch.load(CFG['weight_path'], map_location=device)
    model.load_state_dict(ck.get('state_dict', ck), strict=True)
    print(f'Resumed: {CFG["weight_path"]}')

opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
wu  = 5
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda e: (e+1)/wu if e<wu else
    0.5*(1+np.cos(np.pi*(e-wu)/max(CFG['epochs']-wu,1))))
mse = nn.MSELoss()

best_f1, best_state = -1.0, None
t0 = time.time()

for ep in range(1, CFG['epochs']+1):
    model.train()
    ep_loss, nb = 0.0, 0
    for X, emo, st, eng, att, fat in loader:
        X,emo,st,eng,att,fat = X.to(device),emo.to(device),st.to(device),\
                                eng.to(device),att.to(device),fat.to(device)
        opt.zero_grad()
        o = _fwd(model, X)
        loss = (CFG['emo_loss_w'] * focal_ce(o['emo'], emo, CFG['focal_gamma'],
                                             ew, CFG['label_smooth'], EMOTION_CLASSES)
              + CFG['reg_loss_w'] * (mse(o['st'],st)+mse(o['en'],eng)+
                                     mse(o['at'],att)+mse(o['fa'],fat)))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        ep_loss += loss.item(); nb += 1
    sched.step()
    if ep % 5 == 0 or ep == 1:
        vm = _eval(model, val_split)
        f1 = vm['emotion']['macro_f1']
        print(f'  Ep {ep:3d}/{CFG["epochs"]}  loss={ep_loss/nb:.4f}  '
              f'val_f1={f1:.3f}  stress_rmse={vm["stress"]["rmse"]:.3f}  '
              f't={time.time()-t0:.0f}s', flush=True)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k:v.clone().cpu() for k,v in model.state_dict().items()}
            print(f'    ↑ best val F1={f1:.3f}')

print(f'Done. Total={time.time()-t0:.1f}s  best_val_f1={best_f1:.3f}')

## 10 — Evaluate on test set

In [ ]:
import json
if best_state:
    model.load_state_dict(best_state)
    print(f'Best weights loaded (val F1={best_f1:.3f})')

test_metrics = _eval(model, test_split)
print('\n=== Test metrics ===')
for h, m in test_metrics.items():
    print(f'  {h}: {m}')

annotated = dict(test_metrics)
annotated['_label_provenance'] = {
    'emotion':    'REAL - FER2013/RAF-DB annotations',
    'stress':     'MIXED - WESAD GT + AU/HCI proxy',
    'engagement': 'PROXY - noisy formula; no public GT',
    'attention':  'PROXY - noisy formula; no public GT',
    'fatigue':    'PROXY - noisy formula; no public GT',
}
annotated['_colab_run'] = dict(
    epochs=CFG['epochs'], fer_samples=CFG['fer_samples'],
    n_train=len(train_split['X']), n_val=len(val_split['X']),
    n_test=len(test_split['X']), best_val_f1=best_f1,
    gpu=torch.cuda.get_device_name(0)
)
with open(CFG['metrics_path'], 'w') as f:
    json.dump(annotated, f, indent=2)
print(f'Metrics saved: {CFG["metrics_path"]}')

## 11 — Save weights (local + Drive)

In [ ]:
import os, shutil
os.makedirs(os.path.dirname(CFG['weight_path']), exist_ok=True)
os.makedirs(CFG['drive_dir'], exist_ok=True)

torch.save({
    'state_dict':   model.state_dict(),
    'test_metrics': test_metrics,
    'datasets':     ['FER2013','RAF-DB','WESAD'],
    'n_train': len(train_split['X']),
    'n_val':   len(val_split['X']),
    'n_test':  len(test_split['X']),
    'cfg': CFG,
}, CFG['weight_path'])

sz = os.path.getsize(CFG['weight_path'])/1024**2
print(f'Weights: {CFG["weight_path"]} ({sz:.2f} MB)')

shutil.copy2(CFG['weight_path'],   os.path.join(CFG['drive_dir'], 'tcmt_trained.pt'))
shutil.copy2(CFG['metrics_path'],  os.path.join(CFG['drive_dir'], 'tcmt_eval_metrics.json'))
print(f'Backed up to Drive: {CFG["drive_dir"]}')

## 12 — Download weights to local machine

In [ ]:
from google.colab import files
files.download(CFG['weight_path'])
files.download(CFG['metrics_path'])
print('Files downloaded. Place them in D:\\MHBAP\\ml\\models\\weights\\')

## 13 — Inference sanity check

In [ ]:
model.eval()
dummy = torch.randn(4, 1, 58).to(device)
with torch.no_grad():
    o = _fwd(model, dummy)
preds = o['emo'].argmax(dim=-1).cpu().tolist()
print('Emotion preds (4 samples):', preds)
print('Stress:', o['st'].cpu().numpy().flatten().round(3).tolist())
print('Engagement:', o['en'].cpu().numpy().flatten().round(3).tolist())
assert len(set(preds)) >= 1
print('Sanity check passed.')